# Notebook 2: Experimental Designs, Synchronization Methods, and Control Conditions in Hyperscanning Analysis with HyPyP

In this notebook, we explore advanced methodological aspects of hyperscanning analyses. We will:
- Load dyads from a folder structure (or individual files).
- Apply preprocessing (ICA) to remove artifacts.
- Create pseudo-dyads and surrogate data as control conditions.
- Compute and compare inter-brain connectivity metrics for real and control conditions.

In [ ]:
import os
import glob
import mne
import numpy as np
from collections import OrderedDict

import hypyp.analyses as analyses
import hypyp.viz as viz
import hypyp.prep as prep

# Confirm successful import of libraries
print("Libraries imported successfully.")

## 1. Loading the Data

Here we load the epoch files for two participants from individual files (or subfolders).  
Each file is assumed to contain preprocessed epochs for one participant.  
We equalize the number of epochs between participants for consistent analysis.

In [ ]:
# Load epochs from file paths (adjust the paths as necessary)
epo1 = mne.read_epochs("./data/participant1-epo.fif", preload=True)
epo2 = mne.read_epochs("./data/participant2-epo.fif", preload=True)

# Equalize the number of epochs between participants
mne.epochs.equalize_epoch_counts([epo1, epo2])

# Print summaries for verification
print("Participant 1 Epochs:")
print(epo1)
print("\nParticipant 2 Epochs:")
print(epo2)

# Store dyad data as an array (for later connectivity computations)
dyad = [epo1.get_data(copy=True), epo2.get_data(copy=True)]

## 1.1 Review of Loaded Dyads

The code above loads each dyad from its subfolder.  
Each dyad is stored as a tuple `(epo1, epo2)` where each element is an MNE Epochs object.  
For further analysis, we will process these dyads individually.

In [ ]:
# Display summaries for verification
print("Selected Dyad:")
print("Participant 1 Epochs:", epo1)
print("Participant 2 Epochs:", epo2)

## 2. Preprocessing with ICA

Before computing connectivity, we perform additional preprocessing to remove artifacts such as eye blinks.  
Here we apply ICA using functions from `hypyp.prep`. Adjust parameters (e.g., method, number of components) as needed.

In [ ]:
# Compute ICA for each participant with 15 components
icas = prep.ICA_fit([
    epo1, epo2
],
    n_components=15,
    method='infomax',
    fit_params=dict(extended=True),
    random_state=42
)

# Select the relevant independent components for artefact rejection
cleaned_epochs_ICA = prep.ICA_choice_comp(icas, [epo1, epo2])
print('ICA correction completed.')

# Apply local AutoReject on the ICA-cleaned epochs
cleaned_epochs_AR, dic_AR = prep.AR_local(
    cleaned_epochs_ICA,
    strategy="union",
    threshold=50.0,
    verbose=True
)
print('AutoReject completed.')

# Assign cleaned epochs to individual participant variables
epo1_clean = cleaned_epochs_AR[0]
epo2_clean = cleaned_epochs_AR[1]
print('Preprocessed epochs for both participants are ready.')

# Update dyad with cleaned data for subsequent analysis
dyad_clean = [epo1_clean.get_data(copy=True), epo2_clean.get_data(copy=True)]

## 3. Creating Control Conditions: Pseudo-Dyads and Surrogate Data

To evaluate whether inter-brain synchrony is due to true interaction or common stimulus effects, we generate:
- **Pseudo-Dyads:** by shuffling epochs within a dyad.
- **Surrogate Data:** by time-shuffling segments in each epoch to break temporal alignment while preserving spectral content.

In [ ]:
# Create a pseudo-dyad by shuffling the epochs for one participant.
pseudo_epo1 = epo1_clean.copy()
pseudo_epo2 = epo2_clean.copy()
# Shuffle epochs along the first axis of the data array
np.random.shuffle(pseudo_epo2._data)
print("Pseudo-dyad created by shuffling epochs within the same dyad.")

In [ ]:
def create_surrogate_data(epochs):
    """
    Generate surrogate data by time-shuffling each epoch.
    This breaks true synchrony while preserving frequency characteristics.
    """
    surrogate_epochs = epochs.copy()
    for i in range(len(surrogate_epochs._data)):
        np.random.shuffle(surrogate_epochs._data[i])
    return surrogate_epochs

# Create surrogate epochs for both participants
surrogate_epo1 = create_surrogate_data(epo1_clean)
surrogate_epo2 = create_surrogate_data(epo2_clean)
print("Surrogate data generated for both participants.")

## 4. Advanced Connectivity Analysis

We now compute the analytic signals and connectivity metrics using the HyPyP analyses functions for:
- The real (cleaned) dyad.
- The pseudo-dyad.
- The surrogate data.

We define frequency bands (e.g., two alpha sub-bands) and process the data accordingly.

In [ ]:
# Connectivity Analysis for Real, Pseudo, and Surrogate Conditions

# Extract the sampling rate from the epochs (assumes both participants share the same sfreq)
sampling_rate = epo1_clean.info['sfreq']

# Define frequency bands as an OrderedDict
freq_bands = OrderedDict({
    'Alpha-Low': [7.5, 11],
    'Alpha-High': [11.5, 13]
})

# Prepare data arrays for each condition
data_real   = np.array([epo1_clean.get_data(copy=True), epo2_clean.get_data(copy=True)])
data_pseudo = np.array([pseudo_epo1.get_data(copy=True), pseudo_epo2.get_data(copy=True)])
data_surrog = np.array([surrogate_epo1.get_data(copy=True), surrogate_epo2.get_data(copy=True)])

# Compute analytic signals using FIR filtering and the Hilbert transform
complex_real   = analyses.compute_freq_bands(
    data_real, sampling_rate, freq_bands,
    filter_length=int(sampling_rate),
    l_trans_bandwidth=5.0,
    h_trans_bandwidth=5.0
)

complex_pseudo = analyses.compute_freq_bands(
    data_pseudo, sampling_rate, freq_bands,
    filter_length=int(sampling_rate),
    l_trans_bandwidth=5.0,
    h_trans_bandwidth=5.0
)

complex_surrog = analyses.compute_freq_bands(
    data_surrog, sampling_rate, freq_bands,
    filter_length=int(sampling_rate),
    l_trans_bandwidth=5.0,
    h_trans_bandwidth=5.0
)

# Compute connectivity using the circular correlation ('ccorr') metric and average across epochs
conn_real   = analyses.compute_sync(complex_real,   mode='ccorr', epochs_average=True)
conn_pseudo = analyses.compute_sync(complex_pseudo, mode='ccorr', epochs_average=True)
conn_surrog = analyses.compute_sync(complex_surrog, mode='ccorr', epochs_average=True)

# Determine the number of channels per participant
n_ch = len(epo1_clean.info['ch_names'])

# Slice connectivity matrices to extract inter-brain connectivity
alpha_low_real   = conn_real[:, 0:n_ch, n_ch:2*n_ch]
alpha_low_pseudo = conn_pseudo[:, 0:n_ch, n_ch:2*n_ch]
alpha_low_surrog = conn_surrog[:, 0:n_ch, n_ch:2*n_ch]

# For further analysis, choose the Alpha-Low band values (first frequency band)
values_real   = alpha_low_real
values_pseudo = alpha_low_pseudo
values_surrog = alpha_low_surrog

# Normalize the connectivity matrices using Z-score normalization
C_real   = (values_real   - np.mean(values_real[:]))   / np.std(values_real[:])
C_pseudo = (values_pseudo - np.mean(values_pseudo[:])) / np.std(values_pseudo[:])
C_surrog = (values_surrog - np.mean(values_surrog[:])) / np.std(values_surrog[:])

In [ ]:
# 2D Visualization of Real Dyad Connectivity
viz.viz_2D_topomap_inter(epo1_clean, epo2_clean, C_real[0], threshold='auto', steps=10, lab=True)

In [ ]:
# 2D Visualization of Pseudo-Dyad Connectivity
viz.viz_2D_topomap_inter(pseudo_epo1, pseudo_epo2, C_pseudo[0], threshold='auto', steps=10, lab=True)

In [ ]:
# 2D Visualization of Surrogate Data Connectivity

viz.viz_2D_topomap_inter(surrogate_epo1, surrogate_epo2, C_surrog[0], threshold='auto', steps=10, lab=True)

## 6. Discussion and Future Directions

- **Preprocessing with ICA:**  
  Removing artifacts with ICA (using hypyp.prep) is crucial for reliable connectivity estimates.

- **Control Conditions:**  
  Pseudo-dyads and surrogate data help determine whether the observed synchrony is interaction-driven or an artifact of shared stimuli or environmental factors.

- **Connectivity Metrics:**  
  In this notebook, we used circular correlation (`ccorr`). Future work may compare additional metrics (PLV, coherence, envelope correlation) across conditions.

- **Statistical Analysis:**  
  Subsequent analyses could include statistical comparisons (e.g., permutation tests) between conditions to quantify significant differences.

This notebook highlights how careful experimental design, robust preprocessing, and control conditions contribute to valid hyperscanning analyses.

## Conclusion

In this improved notebook, we have:
- Loaded dyads and applied additional ICA preprocessing to remove artifacts.
- Generated pseudo-dyads and surrogate data as control conditions.
- Computed and normalized connectivity metrics (using circular correlation) for all conditions.
- Visualized the inter-brain connectivity using 2D topographic maps.

These steps lay the groundwork for further investigations into inter-brain synchrony and more advanced statistical analyses.